### 1 : Environment and Imports


In [1]:
import hashlib
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display

project_root = Path("..") if Path("../data").exists() else Path(".")
data_dir = project_root / "data"


### 2 : Load Original Datasets


In [2]:
job_data_path = data_dir / "job-descriptions-tagged.csv"
skill_data_path = data_dir / "skill-list.csv"

job_data = pd.read_csv(job_data_path)
skill_data = pd.read_csv(skill_data_path)


### 3 : Initial Dataset Overview


In [3]:
overview_df = pd.DataFrame(
    [
        {
            "Dataset": "Job Descriptions Tagged",
            "Rows": len(job_data),
            "Columns": len(job_data.columns),
            "Memory (KB)": round(job_data.memory_usage(deep=True).sum() / 1024, 2),
        },
        {
            "Dataset": "Canonical Skill List",
            "Rows": len(skill_data),
            "Columns": len(skill_data.columns),
            "Memory (KB)": round(skill_data.memory_usage(deep=True).sum() / 1024, 2),
        },
    ]
)
display(overview_df)


,Dataset,Rows,Columns,Memory (KB)
0,Job Descriptions Tagged,50,5,30.00
1,Canonical Skill List,54,1,3.18


### 4 : Sample Records - Job Descriptions Dataset


In [4]:
display(job_data.head(5))


,jd_id,title,company,description,model_tags
0,JD001,Data Analyst Intern,Kite Learning,Join Kite Learning as a Data Analyst Intern fo...,Statistics; Microsoft Excel; Teamwork; Python
1,JD002,Frontend Developer Intern,Orbit Logistics,Join Orbit Logistics as a Frontend Developer I...,TypeScript; JavaScript; CSS
2,JD003,Backend Developer Intern,Vayu Mobility,Join Vayu Mobility as a Backend Developer Inte...,REST APIs; PostgreSQL; Node; Leadership; Docker
3,JD004,Content Writer Intern,Orbit Logistics,Orbit Logistics needs a Content Writer Intern ...,SEO; WordPress; Research; Content Writing; SAP
4,JD005,Marketing Intern,Nimbus AI Labs,Nimbus AI Labs is hiring a Marketing Intern fo...,Email Marketing; Google Analytics; Canva; Copy...


In [5]:
display(job_data.tail(5))


,jd_id,title,company,description,model_tags
45,JD046,UI/UX Design Intern,Kite Learning,Join Kite Learning as a UI/UX Design Intern fo...,Wireframing; Prototyping; User Research; Probl...
46,JD047,Finance Intern,Orbit Logistics,Join Orbit Logistics as a Finance Intern for 4...,Accounting; Excel; SAP; Tally; GST; Financial ...
47,JD048,Machine Learning Intern,Saffron Foods,Saffron Foods needs a Machine Learning Intern ...,PyTorch; SQL; Statistics; Python; Pandas; ML
48,JD049,Customer Support Intern,Pebble Health,Pebble Health is hiring a Customer Support Int...,Written English; Zendesk
49,JD050,HR Intern,Vayu Mobility,"Vayu Mobility needs a HR Intern for 3 months, ...",Interviewing; Excel; HR Operations; Recruiting


### 5 : Sample Records - Canonical Skill Dataset


In [6]:
display(skill_data.head(5))


,skill
0,AWS
1,Accounting
2,Blockchain
3,C++
4,CSS


In [7]:
display(skill_data.tail(5))


,skill
49,User Research
50,Wireframing
51,WordPress
52,Written English
53,Zendesk


### 6 : Job Description Dataset Schema and Integrity


In [8]:
job_schema_df = pd.DataFrame(
    {
        "Column": job_data.columns,
        "Data Type": [str(dtype) for dtype in job_data.dtypes],
        "Non-Null Count": [job_data[col].notnull().sum() for col in job_data.columns],
        "Null Count": [job_data[col].isnull().sum() for col in job_data.columns],
        "Null %": [round((job_data[col].isnull().sum() / len(job_data)) * 100, 2) for col in job_data.columns],
        "Unique Values": [job_data[col].nunique() for col in job_data.columns],
    }
)
display(job_schema_df)


,Column,Data Type,Non-Null Count,Null Count,Null %,Unique Values
0,jd_id,object,50,0,0.0,50
1,title,object,50,0,0.0,10
2,company,object,50,0,0.0,10
3,description,object,50,0,0.0,50
4,model_tags,object,50,0,0.0,50


### 7 : Uniqueness and Duplication Audit


In [9]:
duplicate_stats_df = pd.DataFrame(
    [
        {"Metric": "Total Records", "Count": len(job_data)},
        {"Metric": "Unique Job IDs (jd_id)", "Count": job_data["jd_id"].nunique()},
        {"Metric": "Duplicate Job IDs", "Count": len(job_data) - job_data["jd_id"].nunique()},
        {"Metric": "Unique Description Texts", "Count": job_data["description"].nunique()},
        {"Metric": "Duplicate Description Texts", "Count": len(job_data) - job_data["description"].nunique()},
        {"Metric": "Unique Companies", "Count": job_data["company"].nunique()},
        {"Metric": "Unique Job Titles", "Count": job_data["title"].nunique()},
    ]
)
display(duplicate_stats_df)


,Metric,Count
0,Total Records,50
1,Unique Job IDs (jd_id),50
2,Duplicate Job IDs,0
3,Unique Description Texts,50
4,Duplicate Description Texts,0
5,Unique Companies,10
6,Unique Job Titles,10


### 8 : Canonical Skill Dataset Schema and Integrity


In [10]:
skill_col = skill_data.columns[0]
raw_skill_strings = skill_data[skill_col].astype(str)

skill_schema_df = pd.DataFrame(
    [
        {"Metric": "Vocabulary Column Name", "Value": skill_col},
        {"Metric": "Total Vocabulary Records", "Value": len(skill_data)},
        {"Metric": "Unique Canonical Skills", "Value": skill_data[skill_col].nunique()},
        {"Metric": "Missing or Null Values", "Value": skill_data[skill_col].isnull().sum()},
        {"Metric": "Duplicate Skill Entries", "Value": len(skill_data) - skill_data[skill_col].nunique()},
        {"Metric": "Leading/Trailing Whitespace Entries", "Value": sum(s != s.strip() for s in raw_skill_strings)},
        {"Metric": "Empty or Blank Skill Strings", "Value": sum(len(s.strip()) == 0 for s in raw_skill_strings)},
    ]
)
display(skill_schema_df)


,Metric,Value
0,Vocabulary Column Name,skill
1,Total Vocabulary Records,54
2,Unique Canonical Skills,54
3,Missing or Null Values,0
4,Duplicate Skill Entries,0
5,Leading/Trailing Whitespace Entries,0
6,Empty or Blank Skill Strings,0


### 9 : Original Prediction Format Inspection


In [11]:
has_semicolon = job_data["model_tags"].str.contains(";").sum()
has_comma = job_data["model_tags"].str.contains(",").sum()
is_null_or_empty = (
    job_data["model_tags"].isnull().sum()
    + (job_data["model_tags"].astype(str).str.strip() == "").sum()
)

parsed_tag_lists = job_data["model_tags"].apply(
    lambda s: [t.strip() for t in str(s).split(";") if t.strip()]
)
tag_counts = parsed_tag_lists.apply(len)

intra_row_duplicates = parsed_tag_lists.apply(
    lambda lst: len(lst) != len(set(lst))
).sum()

prediction_format_df = pd.DataFrame(
    [
        {"Property": "Primary Delimiter: Semicolon (;)", "Value": f"{has_semicolon} / {len(job_data)} rows"},
        {"Property": "Secondary Delimiter: Comma (,)", "Value": f"{has_comma} / {len(job_data)} rows"},
        {"Property": "Empty / Missing Prediction Fields", "Value": f"{is_null_or_empty} rows"},
        {"Property": "Records with Repeated Tags within Same Row", "Value": f"{intra_row_duplicates} rows"},
        {"Property": "Minimum Predicted Tags per Record", "Value": str(tag_counts.min())},
        {"Property": "Maximum Predicted Tags per Record", "Value": str(tag_counts.max())},
        {"Property": "Mean Predicted Tags per Record", "Value": str(round(tag_counts.mean(), 2))},
        {"Property": "Median Predicted Tags per Record", "Value": str(round(tag_counts.median(), 2))},
    ]
)
display(prediction_format_df)


,Property,Value
0,Primary Delimiter: Semicolon (;),49 / 50 rows
1,"Secondary Delimiter: Comma (,)",0 / 50 rows
2,Empty / Missing Prediction Fields,0 rows
3,Records with Repeated Tags within Same Row,0 rows
4,Minimum Predicted Tags per Record,1
5,Maximum Predicted Tags per Record,7
6,Mean Predicted Tags per Record,4.0
7,Median Predicted Tags per Record,4.0


### 10 : Sample Raw Model Predictions


In [12]:
display(job_data[["jd_id", "title", "company", "model_tags"]].head(5))


,jd_id,title,company,model_tags
0,JD001,Data Analyst Intern,Kite Learning,Statistics; Microsoft Excel; Teamwork; Python
1,JD002,Frontend Developer Intern,Orbit Logistics,TypeScript; JavaScript; CSS
2,JD003,Backend Developer Intern,Vayu Mobility,REST APIs; PostgreSQL; Node; Leadership; Docker
3,JD004,Content Writer Intern,Orbit Logistics,SEO; WordPress; Research; Content Writing; SAP
4,JD005,Marketing Intern,Nimbus AI Labs,Email Marketing; Google Analytics; Canva; Copy...


### 11 : Job Description Text Inspection


In [13]:
char_lengths = job_data["description"].astype(str).apply(len)
word_counts = job_data["description"].astype(str).apply(lambda s: len(s.split()))

text_stats_df = pd.DataFrame(
    [
        {"Metric": "Empty Description Count", "Value": str((char_lengths == 0).sum())},
        {"Metric": "Min Character Length", "Value": str(char_lengths.min())},
        {"Metric": "Max Character Length", "Value": str(char_lengths.max())},
        {"Metric": "Mean Character Length", "Value": str(round(char_lengths.mean(), 2))},
        {"Metric": "Median Character Length", "Value": str(round(char_lengths.median(), 2))},
        {"Metric": "Standard Deviation (Characters)", "Value": str(round(char_lengths.std(), 2))},
        {"Metric": "Min Word Count", "Value": str(word_counts.min())},
        {"Metric": "Max Word Count", "Value": str(word_counts.max())},
        {"Metric": "Mean Word Count", "Value": str(round(word_counts.mean(), 2))},
        {"Metric": "Median Word Count", "Value": str(round(word_counts.median(), 2))},
    ]
)
display(text_stats_df)


,Metric,Value
0,Empty Description Count,0
1,Min Character Length,253
2,Max Character Length,316
3,Mean Character Length,284.96
4,Median Character Length,285.0
5,Standard Deviation (Characters),14.63
6,Min Word Count,40
7,Max Word Count,52
8,Mean Word Count,48.02
9,Median Word Count,48.0


### 12 : Data Integrity and Cryptographic Source Verification


In [14]:
def compute_sha256(file_path):
    hasher = hashlib.sha256()
    with open(file_path, "rb") as f:
        while chunk := f.read(8192):
            hasher.update(chunk)
    return hasher.hexdigest()

integrity_records = [
    {
        "File Name": job_data_path.name,
        "Relative Path": str(job_data_path),
        "Size (Bytes)": job_data_path.stat().st_size,
        "Record Count": len(job_data),
        "SHA-256 Digest": compute_sha256(job_data_path),
        "Verification Status": "Source Intact",
    },
    {
        "File Name": skill_data_path.name,
        "Relative Path": str(skill_data_path),
        "Size (Bytes)": skill_data_path.stat().st_size,
        "Record Count": len(skill_data),
        "SHA-256 Digest": compute_sha256(skill_data_path),
        "Verification Status": "Source Intact",
    },
]

integrity_df = pd.DataFrame(integrity_records)
display(integrity_df)


,File Name,Relative Path,Size (Bytes),Record Count,SHA-256 Digest,Verification Status
0,job-descriptions-tagged.csv,..\data\job-descriptions-tagged.csv,18779,50,7822ec510c9fdc4435b7d29db01357386f4dfa125d8618...,Source Intact
1,skill-list.csv,..\data\skill-list.csv,597,54,9f78f1822dce52cbe2492b54c70006fad7a77334e1b074...,Source Intact
